In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
hf_home = os.getenv("HF_HOME")
if hf_home:
    os.environ["HF_HOME"] = hf_home
from tqdm import tqdm
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import wandb
device = "mps"
np.set_printoptions(threshold=np.inf)
import matplotlib.pyplot as plt

/opt/miniconda3/envs/axis_rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model 준비

In [2]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices="mps") 

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 267721.53it/s]


# Dataset 준비

In [33]:
marco_data = load_dataset("microsoft/ms_marco", "v1.1", split ="train", streaming=True)
passage_embedding = np.load("data/ms_marco_train_p_embedding.npy")
passage_embedding_structued = []

total = 0

for data in marco_data:
    num_passage = len(data['passages']['is_selected'])
    
    passage_embedding_structued.append(passage_embedding[:num_passage])
    passage_embedding = passage_embedding[num_passage:]

# Train Query Regularization

In [13]:
train_q_embeddings = np.load("data/ms_marco_train_q_embedding.npy")
train_q_mean = np.mean(train_q_embeddings, axis = 0)
train_q_std = np.std(train_q_embeddings, axis = 0)

# Test w/ Validation Set

In [4]:
marco_val_data = load_dataset("microsoft/ms_marco", "v1.1", split ="validation", streaming=True)

In [ ]:
marco_val_queries = [data['query'] for data in marco_val_data]

marco_val_passages_num = []
marco_val_selected_passage = [] 

for data in marco_val_data:
    marco_val_passages_num.append(len(data['passages']['passage_text']))


In [21]:
for data in marco_val_data:
    print(data)
    break


{'answers': ['Approximately $15,000 per year.'], 'passages': {'is_selected': [1, 0, 0, 0, 0, 0], 'passage_text': ['The average Walgreens salary ranges from approximately $15,000 per year for Customer Service Associate / Cashier to $179,900 per year for District Manager. Average Walgreens hourly pay ranges from approximately $7.35 per hour for Laboratory Technician to $68.90 per hour for Pharmacy Manager. Salary information comes from 7,810 data points collected directly from employees, users, and jobs on Indeed.', 'The average revenue in 2011 of a Starbuck Store was $1,078,000, up  from $1,011,000 in 2010.    The average ticket (total purchase) at domestic Starbuck stores in  No … vember 2007 was reported at $6.36.    In 2008, the average ticket was flat (0.0% change).', 'In fiscal 2014, Walgreens opened a total of 184 new locations and acquired 84 locations, for a net decrease of 273 after relocations and closings. How big are your stores? The average size for a typical Walgreens is a

In [9]:
val_p_embedding = np.load("data/ms_marco_val_p_embedding.npy")
val_p_embedding_structured = []

for n in marco_val_passages_num:
    val_p_embedding_structured.append(val_p_embedding[:n])
    val_p_embedding = val_p_embedding[n:]

val_q_embedding = np.load("data/ms_marco_val_q_embedding.npy")

In [11]:
print(len(val_p_embedding_structured))
print(len(val_q_embedding))

10047
10047


### Full Score 계산

In [65]:
n_dim = 100
mercy = 5

In [71]:
# Full Score 계산
right_count = 0 

for i, data in tqdm(enumerate(marco_val_data), total=len(val_q_embedding)):    
    selected_passage = data['passages']['is_selected']
    if(1 not in selected_passage): continue

    ans_idx = data['passages']['is_selected'].index(1)

    # print((val_p_embedding_structured[i] @ val_q_embedding[i]))
    ranking = np.argsort((val_p_embedding_structured[i] @ val_q_embedding[i]))
    if(ans_idx in ranking[-mercy:]):
        right_count += 1

print(right_count)

100%|██████████| 10047/10047 [00:01<00:00, 5306.68it/s]

8509


### Partial Score 계산

In [72]:
right_count = 0
for i, data in tqdm(enumerate(marco_val_data), total=len(val_q_embedding)):    
    selected_passage = data['passages']['is_selected']
    if(1 not in selected_passage): continue

    ans_idx = data['passages']['is_selected'].index(1)

    val_q_embedding_normalized = (val_q_embedding[i] - train_q_mean) / train_q_std
    dims_selected = np.argsort(np.abs(val_q_embedding_normalized))[-n_dim:]

    ranking = np.argsort((val_p_embedding_structured[i][:,dims_selected] @ val_q_embedding[i][dims_selected]))
    if(ans_idx in ranking[-mercy:]):
        right_count += 1

print(right_count)

100%|██████████| 10047/10047 [00:02<00:00, 4658.07it/s]

8370


### Random Score 계산

In [73]:
import random
right_count = 0

for i, data in tqdm(enumerate(marco_val_data), total=len(val_q_embedding)):    
    selected_passage = data['passages']['is_selected']
    if(1 not in selected_passage): continue

    ans_idx = data['passages']['is_selected'].index(1)

    rndm_k_idx = random.sample(range(1024), n_dim)

    ranking = np.argsort((val_p_embedding_structured[i][:,rndm_k_idx] @ val_q_embedding[i][rndm_k_idx]))
    if(ans_idx in ranking[-mercy:]):
        right_count += 1

print(right_count)

100%|██████████| 10047/10047 [00:02<00:00, 4805.65it/s]

8019
